# 第1章 ハンズオン：bitsandbytes / NF4 で QLoRA 微調整

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shintaro-Abe/llm-quantization-dojo/blob/main/notebooks/01_bitsandbytes_qlora.ipynb)

| 項目 | 内容 |
| :-- | :-- |
| 所要時間 | 約 15〜25 分（無料 T4・初回はモデルDL含む） |
| 必要 GPU | 無料 Colab T4（約 15GB VRAM）で完走する設計 |
| 既定モデル | `HuggingFaceTB/SmolLM2-1.7B`（Apache-2.0 / 非ゲート / safetensors / `trust_remote_code` 不要） |
| やること | NF4 で 4bit ロード → 小データで QLoRA 微調整 → 推論の前後比較 |
| 最終確認日 | 2026-06-20 |

!!! note
    **このNotebookの検証状況**：作者により**静的検証（nbformat妥当・各コードセルのPython構文）＋ロジック検証**まで実施済みです。**T4実機での「Run all」完走は、あなたの環境で確認してください**（無料GPUは時期により挙動が変わるため）。

## このNotebookでやること

「**量子化（4bitロード）**」と「**微調整（QLoRA）**」は別ステップです。本Notebookでは両方を順に体験します。

1. **NF4 で 4bit ロード** … 1.7B モデルを軽くして T4 に載せる
2. **QLoRA 微調整** … 凍結したモデルの上に LoRA アダプタだけを学習
3. **前後比較** … 同じ質問に対する応答が、微調整でどう変わるかを目で見る

### デモのねらい（効果が一目で分かるタスク）

十数件の小データで「**どんな質問にも、語尾を必ず『〜だミャ。』にして1文で答える**」ように微調整します。

- **微調整前**：語尾はバラバラ・冗長
- **微調整後**：『〜だミャ。』調の短い応答 → 変化が一目瞭然

!!! warning "正直な注意：これは"効果を見せるための極端なデモ"です"
    十数件しか使わないため、モデルはこのフォーマットを**過学習（丸暗記）**します。デモでは好都合ですが、**実務の微調整は数百〜数千件の多様なデータ**と、検証データでの評価が必要です。ここでは「QLoRAでモデルの振る舞いを変えられる」ことの体感が目的です。


In [ ]:
# セットアップ：量子化・学習に必要なライブラリを導入
# 方針：torch は Colab 既定に従い、量子化/学習系のみ緩めに下限指定（pin しすぎると Colab の torch と衝突しうる）
!pip install -q -U \
    "transformers>=4.44" \
    "peft>=0.12" \
    "accelerate>=0.33" \
    "datasets>=2.20" \
    "bitsandbytes>=0.43"


In [ ]:
# GPU が割り当てられているか確認（Tesla T4 と VRAM が見えれば OK）
# 出ない場合：メニュー「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」
!nvidia-smi


In [ ]:
# インポートと再現性のための seed 固定
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "GPU が見つかりません。ランタイムのタイプを T4 GPU に変更してください。"
print("CUDA:", torch.cuda.get_device_name(0))


## ステップ1：NF4 で 4bit ロード

`BitsAndBytesConfig` で **NF4・二重量子化**を指定して、モデルを 4bit で読み込みます。
T4 は bf16 に非対応（Turing 世代）のため、計算用 dtype は **fp16** を使います。


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "HuggingFaceTB/SmolLM2-1.7B"  # OOM 時は "TinyLlama/TinyLlama-1.1B-Chat-v1.0" に差し替え

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4(Turing) は bf16 非対応のため fp16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
print(model.get_memory_footprint() / 1e9, "GB （4bit ロード後の概算メモリ）")


## ステップ2：微調整「前」の応答を見る

まず素のモデル（4bit ロードのみ・未微調整）の応答を確認します。
共通のプロンプト書式を決め、後の比較でも同じ書式を使います。


In [ ]:
PROMPT_TMPL = "### 質問: {q}\n### 回答: "

# 学習には使わない「未知の質問」で前後比較する
test_questions = ["好きな季節は？", "プログラミングのコツは？", "週末の予定は？"]

def generate(question, max_new_tokens=40):
    text = PROMPT_TMPL.format(q=question)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

print("=== 微調整前 ===")
for q in test_questions:
    print(f"Q: {q}\nA: {generate(q)}\n")


## ステップ3：小さな学習データ（直書き）

外部依存ゼロ・即再現・著作権安全のため、学習データは**この場に直書き**します。
すべて「質問 → 『〜だミャ。』調の1文」になっています。


In [ ]:
from datasets import Dataset

train_data = [
    {"q": "空はなぜ青いの？", "a": "光の散乱で青がよく見えるからだミャ。"},
    {"q": "おすすめの朝ごはんは？", "a": "卵かけご飯が手軽でうまいミャ。"},
    {"q": "運動は何がいい？", "a": "毎日の散歩が続けやすくて一番だミャ。"},
    {"q": "猫は何が好き？", "a": "日なたで昼寝するのが大好きだミャ。"},
    {"q": "勉強のコツは？", "a": "短い時間に区切って集中するといいミャ。"},
    {"q": "雨の日の過ごし方は？", "a": "家で本を読むのがおすすめだミャ。"},
    {"q": "コーヒーと紅茶どっち？", "a": "気分で選べばどっちもいいミャ。"},
    {"q": "早起きのコツは？", "a": "前の晩に早く寝るのが近道だミャ。"},
    {"q": "旅行はどこがいい？", "a": "近場の温泉でのんびりするといいミャ。"},
    {"q": "好きな食べ物は？", "a": "焼き魚がしみじみおいしいミャ。"},
    {"q": "集中できないときは？", "a": "一度立って深呼吸するといいミャ。"},
    {"q": "おすすめの趣味は？", "a": "写真を撮ると毎日が楽しくなるミャ。"},
    {"q": "夜眠れないときは？", "a": "スマホを置いて部屋を暗くするといいミャ。"},
    {"q": "節約のコツは？", "a": "使う前に一呼吸おいて考えるといいミャ。"},
    {"q": "元気が出ないときは？", "a": "あたたかいものを飲んで休むといいミャ。"},
]

def to_text(ex):
    return {"text": PROMPT_TMPL.format(q=ex["q"]) + ex["a"] + tokenizer.eos_token}

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, max_length=128)

raw = Dataset.from_list(train_data).map(to_text)
ds = raw.map(tokenize, remove_columns=raw.column_names)
print(f"学習サンプル数: {len(ds)}")
print("例:", raw[0]["text"])


## ステップ4：QLoRA の設定と学習

4bit のモデルを凍結し、**LoRA アダプタだけ**を学習します（これが QLoRA）。
`prepare_model_for_kbit_training` で 4bit 学習の準備をしてから、LoRA を適用します。


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # SmolLM2 は Llama 系アーキテクチャのため、典型的な対象モジュールを指定
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.config.use_cache = False  # 学習中は無効化（推論時に再度有効化）
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="out_qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=30,          # 小データを覚え込ませる（デモのため多め）
    learning_rate=2e-4,
    fp16=True,                    # T4 は fp16
    logging_steps=10,
    save_strategy="no",
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()


## ステップ5：微調整「後」の応答（前後比較）

学習に使っていない**同じ未知の質問**で、応答がどう変わったかを見ます。
語尾が『〜だミャ。』調の短い応答になっていれば成功です。


In [ ]:
model.config.use_cache = True  # 推論用に再度有効化

print("=== 微調整後 ===")
for q in test_questions:
    print(f"Q: {q}\nA: {generate(q)}\n")


## まとめと振り返り

- **量子化（4bitロード）** と **微調整（QLoRA）** は別ステップだと体感できました。
- 十数件の小データでも、LoRA アダプタの学習だけで**応答の振る舞いを変えられる**ことを確認しました。
- ただしこれは**過学習を利用した極端なデモ**です。実務では多様な十分量のデータと評価が必要です。

### 理解度を確認しよう

- 「4bit でロードした」と「QLoRA で微調整した」の違いを説明できますか？
- 学習データを増やす／減らすと、前後の差はどう変わりそうですか？
- OOM が出たら、どこを変えれば完走できますか？（ヒント：モデル／バッチ／系列長）

### 次のアクション

- 学んだことを **chapter-task Issue** に記録しましょう（進め方は進捗管理の手順書で案内します）。
- 別モデル（例：`TinyLlama-1.1B-Chat`、発展で `Qwen2.5-1.5B`）でも同じ手順が通るか試してみましょう。
- 概念の復習：[第1章 概念](../learn/chapters/01-bitsandbytes-qlora/index.md) ／ [公式リンク集](../learn/chapters/01-bitsandbytes-qlora/references.md)
